# Indian Kanoon API Demo (All Official Endpoints)

This notebook demonstrates the official Indian Kanoon API endpoints:

1. Search: `/search/?formInput=<query>&pagenum=<pagenum>`
2. Document: `/doc/<docid>/`
3. Court Copy (Original): `/origdoc/<docid>/`
4. Document Fragments: `/docfragment/<docid>/?formInput=<query>`
5. Document Meta: `/docmeta/<docid>/`

It fetches real results using your API token and prints stakeholder-friendly outputs.


In [37]:
!pip -q install requests pandas lxml beautifulsoup4 > /dev/null

In [38]:
import os
import json
import textwrap
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

In [ ]:
BASE_URL = "https://api.indiankanoon.org"

IK_TOKEN = "<input your token here, and remove the angle brackets>"

assert IK_TOKEN and IK_TOKEN != "PASTE_YOUR_TOKEN_HERE", "Please set IK_TOKEN before running."

HEADERS = {
    "Authorization": f"Token {IK_TOKEN}",
    "Accept": "application/json",
    "User-Agent": "StakeholderDemo/1.0 (Colab)"
}

print("Token configured. Base URL:", BASE_URL)


Token configured. Base URL: https://api.indiankanoon.org


In [40]:
def ik_request(method: str, path: str, params=None, data=None, timeout=60):
    url = BASE_URL.rstrip("/") + "/" + path.lstrip("/")
    method = method.upper().strip()

    if method == "GET":
        r = requests.get(url, headers=HEADERS, params=params, timeout=timeout)
    elif method == "POST":
        post_headers = dict(HEADERS)
        post_headers["Content-Type"] = "application/x-www-form-urlencoded"
        r = requests.post(url, headers=post_headers, params=params, data=data, timeout=timeout)
    else:
        raise ValueError("method must be GET or POST")

    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code} for {r.url}\nResponse:\n{r.text[:2000]}")

    ct = (r.headers.get("Content-Type") or "").lower()
    if "application/json" in ct:
        return r.json()
    return r.text


def ik_get(path: str, params=None, timeout=60):
    return ik_request("GET", path, params=params, timeout=timeout)


def ik_post(path: str, params=None, data=None, timeout=60):
    return ik_request("POST", path, params=params, data=data, timeout=timeout)


def preview_text(text, max_chars=1500):
    if text is None:
        print("(None)")
        return
    s = str(text).strip()
    if len(s) > max_chars:
        s = s[:max_chars] + "\n\n...[truncated]..."
    print(s)


def html_to_text(html: str) -> str:
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for br in soup.find_all("br"):
        br.replace_with("\n")
    return soup.get_text("\n")


def save_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved JSON to: {filename}")


def save_text(text, filename):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved text to: {filename}")

In [41]:
def search_cases(query: str, pagenum: int = 0, doctypes: str = None,
                fromdate: str = None, todate: str = None,
                title: str = None, cite: str = None,
                author: str = None, bench: str = None,
                maxcites: int = None, maxpages: int = None):
    payload = {
        "formInput": query,
        "pagenum": str(pagenum),
    }
    if doctypes: payload["doctypes"] = doctypes
    if fromdate: payload["fromdate"] = fromdate
    if todate: payload["todate"] = todate
    if title: payload["title"] = title
    if cite: payload["cite"] = cite
    if author: payload["author"] = author
    if bench: payload["bench"] = bench
    if maxcites is not None: payload["maxcites"] = str(maxcites)
    if maxpages is not None: payload["maxpages"] = str(maxpages)

    try:
        return ik_post("/search/", data=payload)
    except RuntimeError as e:
        if "GET\" not allowed" in str(e) or "405" in str(e):
            return ik_get("/search/", params=payload)
        raise

QUERY = 'Article 21 privacy'
PAGE = 0

search_res = search_cases(QUERY, PAGE)

print("Search response keys:", list(search_res.keys())[:20])
save_json(search_res, "search_response.json")

def extract_docids(search_json):
    docids = []
    candidates = []
    for k in ["docs", "results", "docids", "documents"]:
        if k in search_json:
            candidates.append(search_json[k])

    for c in candidates:
        if isinstance(c, list):
            for item in c:
                if isinstance(item, dict):
                    for dk in ["docid", "tid", "id"]:
                        if dk in item:
                            docids.append(str(item[dk]))
                            break
                else:
                    docids.append(str(item))
        elif isinstance(c, str):
            docids.extend([x.strip() for x in c.split(",") if x.strip()])

    if "docid" in search_json:
        docids.append(str(search_json["docid"]))

    seen, out = set(), []
    for d in docids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out

docids = extract_docids(search_res)
print(f"\n Query: {QUERY}")
print(f"Extracted docids (up to 10): {docids[:10]}")

if not docids:
    print("\n Could not auto-extract docids from this response shape.")
    preview_text(json.dumps(search_res, ensure_ascii=False), max_chars=2000)

Search response keys: ['categories', 'docs', 'found', 'encodedformInput']
Saved JSON to: search_response.json

 Query: Article 21 privacy
Extracted docids (up to 10): ['232602', '107953018', '174513855', '91938676', '103640961', '10892972', '144183758', '143202244', '37517217', '973841']


In [42]:
DOCID = docids[0]

print("Selected DOCID:", DOCID)

Selected DOCID: 232602


In [43]:
def get_document(docid: str):
    return ik_post(f"/doc/{docid}/", data={})

doc_res = get_document(DOCID)
save_json(doc_res, f"doc_{DOCID}.json")

print("/doc response keys:", list(doc_res.keys())[:30])

def extract_document_text(doc_json):
    for key in ["doc", "content", "text", "judgement", "html"]:
        if key in doc_json and isinstance(doc_json[key], str):
            return doc_json[key]
    for key in ["data", "result"]:
        if key in doc_json and isinstance(doc_json[key], dict):
            for k2 in ["doc", "content", "text", "html"]:
                if k2 in doc_json[key] and isinstance(doc_json[key][k2], str):
                    return doc_json[key][k2]
    return None

doc_html = extract_document_text(doc_res)
doc_plain = html_to_text(doc_html) if doc_html else ""

print("\n Parsed judgment preview:")
preview_text(doc_plain, max_chars=2000)

save_text(doc_plain, f"doc_{DOCID}_plain.txt")


Saved JSON to: doc_232602.json
/doc response keys: ['tid', 'publishdate', 'title', 'doc', 'numcites', 'numcitedby', 'docsource', 'citetid', 'divtype', 'relatedqs', 'cats', 'courtcopy', 'query_alert']

 Parsed judgment preview:
Sunkara Satyanarayana vs State Of Andhra Pradesh, Home ... on 15 October, 1999


Equivalent citations: 2000(1)ALD(CRI)117, 1999(6)ALT249, 2000CRILJ1297


ORDER
 

V.V.S. Rao, J.
 


1. The petitioner is a Truck driver. He is in private service. He has been residing in Gudivada town of Krishna District eking out his livelihood by working as a lorry driver for the last two years. He has a family depending on him. In this writ petition he has prayed this Court for declaration that the action of the respondents in maintaining history sheet No. 615 in II Town Police Station, Gudivada against the petitioner as illegal and unconstitutional and consequential direction to the respondents to close the history sheet of the petitioner.



2. The facts in this case are not in

In [44]:
def get_origdoc(docid: str):
    return ik_post(f"/origdoc/{docid}/", data={})

orig_res = get_origdoc(DOCID)

if isinstance(orig_res, dict):
    save_json(orig_res, f"origdoc_{DOCID}.json")
    orig_html = extract_document_text(orig_res) or json.dumps(orig_res, ensure_ascii=False)
else:
    orig_html = str(orig_res)
    save_text(orig_html, f"origdoc_{DOCID}_raw.html")

orig_plain = html_to_text(orig_html)

print("\n Original court copy preview:")
preview_text(orig_plain, max_chars=2000)

save_text(orig_plain, f"origdoc_{DOCID}_plain.txt")


Saved JSON to: origdoc_232602.json

 Original court copy preview:
{"errmsg": "Original document not available"}
Saved text to: origdoc_232602_plain.txt


In [45]:
def get_doc_fragments(docid: str, query: str):
    payload = {"formInput": query}
    return ik_post(f"/docfragment/{docid}/", data=payload)

FRAG_QUERY = "right to privacy"

frag_res = get_doc_fragments(DOCID, FRAG_QUERY)
save_json(frag_res, f"docfragment_{DOCID}.json")

print("/docfragment response keys:", list(frag_res.keys())[:30])

def extract_fragments(frag_json):
    for key in ["fragments", "results", "docs", "snippets"]:
        if key in frag_json and isinstance(frag_json[key], list):
            return frag_json[key]
    for key in ["data", "result"]:
        if key in frag_json and isinstance(frag_json[key], dict):
            for k2 in ["fragments", "results", "snippets"]:
                if k2 in frag_json[key] and isinstance(frag_json[key][k2], list):
                    return frag_json[key][k2]
    return []

fragments = extract_fragments(frag_res)

print(f"\n Fragment query: {FRAG_QUERY}")
print(f"Fragments found: {len(fragments)}")

for i, fr in enumerate(fragments[:5], start=1):
    if isinstance(fr, dict):
        txt = fr.get("fragment") or fr.get("text") or fr.get("snippet") or fr.get("content") or str(fr)
    else:
        txt = str(fr)
    print(f"\n--- Fragment {i} ---")
    preview_text(html_to_text(txt), max_chars=800)

Saved JSON to: docfragment_232602.json
/docfragment response keys: ['headline', 'title', 'formInput', 'tid']

 Fragment query: right to privacy
Fragments found: 0


In [46]:
def get_docmeta(docid: str):
    return ik_post(f"/docmeta/{docid}/", data={})

meta_res = get_docmeta(DOCID)
save_json(meta_res, f"docmeta_{DOCID}.json")

print("/docmeta response keys:", list(meta_res.keys())[:40])

def meta_summary(meta):
    fields = ["title", "headline", "court", "date", "bench", "citation", "judges", "acts", "doctype", "url"]
    out = {k: meta[k] for k in fields if k in meta}
    if not out:
        for key in ["data", "result"]:
            if key in meta and isinstance(meta[key], dict):
                out = {k: meta[key][k] for k in fields if k in meta[key]}
                if out:
                    break
    return out

print("\n Metadata summary:")
print(json.dumps(meta_summary(meta_res), indent=2, ensure_ascii=False))


Saved JSON to: docmeta_232602.json
/docmeta response keys: ['tid', 'publishdate', 'doctype', 'relurl', 'caseno', 'numcites', 'numcitedby', 'title']

 Metadata summary:
{
  "title": "Sunkara Satyanarayana vs State Of Andhra Pradesh, Home ... on 15 October, 1999",
  "doctype": "Andhra HC (Pre-Telangana)"
}


In [47]:
filtered = search_cases(
    query='privacy "Article 21"',
    pagenum=0,
    doctypes="supremecourt",
    fromdate="01-01-2010",
    todate="01-01-2025",
    maxcites=10,
    maxpages=2
)

save_json(filtered, "search_filtered_supremecourt.json")
print("Filtered search keys:", list(filtered.keys())[:10])
print("Found:", filtered.get("found"))
print("Docs returned:", len(filtered.get("docs", [])))

df_f = pd.DataFrame(filtered.get("docs", []))
display(df_f.head(10))


Saved JSON to: search_filtered_supremecourt.json
Filtered search keys: ['categories', 'docs', 'found', 'encodedformInput']
Found: 1 - 20 of 2615
Docs returned: 20


,tid,catids,doctype,publishdate,authorid,bench,title,numcites,numcitedby,headline,docsize,fragment,docsource,citation,cites,author,authorEncoded
0,232602,"[46, 0, 4, 1]",1009,1999-10-15,NaN,None,Sunkara Satyanarayana vs State Of Andhra Prade...,85,57,"rights under <b>Articles</b> 14 , 19 and <...",144320,True,Andhra HC (Pre-Telangana),2000(1)ALD(CRI)117,"[{'tid': 1199182, 'title': 'Article 21 in Cons...",NaN,NaN
1,107953018,[46],1004,2019-10-22,1409.0,"[1409, 453]",Vinit Kumar vs Central Burau Of Investigation ...,25,1,restrained under <b>Article</b> <b>21</b> re...,93688,True,Bombay High Court,AIRONLINE 2019 BOM 1117,"[{'tid': 1199182, 'title': 'Article 21 in Cons...",R More,r-more
2,174513855,"[46, 260, 1]",1013,2022-01-03,1161.0,[1161],"S.K.Pavithran vs Laisy Santhosh on 3 January, ...",16,0,regards the concept of right to <b>privacy</b>...,95552,True,Kerala High Court,NaN,"[{'tid': 1199182, 'title': 'Article 21 in Cons...",P B Kumar,p-b-kumar
3,91938676,"[46, 232]",1000,2017-08-24,429.0,"[971, 706, 429, 1194, 1730, 187, 1266, 1127, 1...",Justice K.S.Puttaswamy(Retd) And Anr. vs Union...,294,384,<b>Articles</b> 19 and <b>21</b> to be one ...,1223042,True,Supreme Court of India,AIR 2017 SUPREME COURT 4161,"[{'tid': 1199182, 'title': 'Article 21 in Cons...",D Y Chandrachud,d-y-chandrachud
4,10892972,"[46, 4, 44]",1033,2022-07-15,NaN,None,Udathu Suresh vs The State Of Andhra Pradesh A...,53,2,restrained\n under <b>Article</b> <b>21</b>...,93538,True,Andhra Pradesh High Court - Amravati,NaN,"[{'tid': 1199182, 'title': 'Article 21 in Cons...",NaN,NaN
5,144183758,None,1007,2025-07-02,1341.0,[1341],P.Kishore vs The Secretary To Government on 2 ...,61,0,aspect\n\n of personal lib...,232103,True,Madras High Court,NaN,"[{'tid': 1445510, 'title': 'Section 5(2) in Th...",N A Venkatesh,n-a-venkatesh
6,143202244,"[41, 4, 381]",1000,2020-10-29,1730.0,"[544, 1730]",Tofan Singh vs The State Of Tamil Nadu on 29 O...,360,1439,Tofan Singh vs The State Of Tamil Nadu on 29 ...,814708,True,Supreme Court of India,AIR 2020 SUPREME COURT 5592,"[{'tid': 1727139, 'title': 'The Narcotic Drugs...",R F Nariman,r-f-nariman
7,973841,[46],1000,1996-12-18,1734.0,"[1734, 2201]",People`S Union For Civilliberties ... vs The U...,22,330,enshrined under <b>Article</b> <b>21</b> o...,56963,True,Supreme Court of India,NaN,"[{'tid': 1445510, 'title': 'Section 5(2) in Th...",K Singh,kuldip-singh
8,31276692,[46],1000,1996-12-18,1734.0,"[1734, 2201]",People'S Union Of Civil Liberties ... vs Union...,20,181,Parikh vehemently contended that right to <b>p...,51579,True,Supreme Court of India,AIR1997SC568,"[{'tid': 1445510, 'title': 'Section 5(2) in Th...",K Singh,kuldip-singh
9,698472,"[46, 662, 44]",1000,2008-09-01,1527.0,"[1527, 1893, 1935]",State Of Maharashtra vs Bharat Shanti Lal Shah...,50,95,enriched under\n\n <b>Article</b> <b>21</b>...,78229,True,Supreme Court of India,2008 AIR SCW 6431,"[{'tid': 100648, 'title': 'Section 16 in The I...",M Sharma,m-sharma


In [48]:
doc_res = get_document(DOCID)
orig_res = get_origdoc(DOCID)

doc_plain = html_to_text(extract_document_text(doc_res) or "")
orig_plain = html_to_text(extract_document_text(orig_res) or (orig_res if isinstance(orig_res, str) else ""))

print("\n===== /doc (parsed) preview =====")
preview_text(doc_plain, max_chars=1200)

print("\n===== /origdoc (court copy) preview =====")
preview_text(orig_plain, max_chars=1200)



===== /doc (parsed) preview =====
Sunkara Satyanarayana vs State Of Andhra Pradesh, Home ... on 15 October, 1999


Equivalent citations: 2000(1)ALD(CRI)117, 1999(6)ALT249, 2000CRILJ1297


ORDER
 

V.V.S. Rao, J.
 


1. The petitioner is a Truck driver. He is in private service. He has been residing in Gudivada town of Krishna District eking out his livelihood by working as a lorry driver for the last two years. He has a family depending on him. In this writ petition he has prayed this Court for declaration that the action of the respondents in maintaining history sheet No. 615 in II Town Police Station, Gudivada against the petitioner as illegal and unconstitutional and consequential direction to the respondents to close the history sheet of the petitioner.



2. The facts in this case are not in dispute. A crime was registered against the petitioner Under 
Section 379
 of Indian Penal Code, 1860 (
I.P.C
. for short) in 1972 and also in 1976. In both the cases, he was convicted and se

In [49]:
QUESTION = "What has the Supreme Court said about privacy as a fundamental right under Article 21?"

sr = search_cases("privacy fundamental right Article 21", pagenum=0, doctypes="supremecourt", maxpages=1)
docs = sr.get("docs", [])
if not docs:
    raise RuntimeError("No docs found for demo query. Try changing the query.")

demo_docid = str(docs[0].get("docid"))
print("Demo docid:", demo_docid)
print("Title:", docs[0].get("title", "(no title)"))

# # Step 2: Pull fragments relevant to the stakeholder question
# fr = get_doc_fragments(demo_docid, "privacy fundamental right Article 21")
# frags = extract_fragments(fr)

# print("\n Supporting fragments (top 5):")
# for i, f in enumerate(frags[:5], start=1):
#     txt = f.get("fragment") if isinstance(f, dict) else str(f)
#     print(f"\n--- Evidence {i} ---")
#     preview_text(html_to_text(txt), max_chars=700)

# print("\n Stakeholder takeaway: We can answer questions and show evidence snippets used to justify the answer.")


Demo docid: None
Title: Sunkara Satyanarayana vs State Of Andhra Pradesh, Home ... on 15 October, 1999
